# FGPT: Interactive Exploration

This notebook provides hands-on examples of using the FGPT `Processor`, `Extractor`, `Navigator`, etc classes for Fortran code parsing, transformation, and analysis. 

In [1]:
%load_ext autoreload
%autoreload 2
from fparser.two import Fortran2003 as F23
from fparser.two import Fortran2008 as F28
from fparser.two.utils import walk
import os

In [2]:
# change to the Fgpt directory
%cd /home/kardaneh/Fgpt

/home/kardaneh/Fgpt


/home/kardaneh/Fgpt/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# Processor class
from fgpt.core.frontend import Processor
from fgpt.core.common import Logger
logger = Logger()
processor = Processor(logger=Logger())

In [4]:
# A simple Fortran program/module/proceduer as a string
code = """
subroutine increment_counter(value)
    integer, intent(in) :: value
    global_counter = global_counter + value
end subroutine increment_counter
"""
subroutine_tree = processor.parse_fortran_string(code)
processor.logger.info(subroutine_tree)

[INFO] Successfully parsed string!

[INFO] 
SUBROUTINE increment_counter(value)
  INTEGER, INTENT(IN) :: value
  global_counter = global_counter + value
END SUBROUTINE increment_counter

In [5]:
# A piece of Fortran code as string
code = """
minval_value = HUGE(0.0)
minval_index = 1
DO i = 1, N
  IF (A(i) .LT. minval_value) THEN
    minval_value = A(i)
    minval_index = i
  END IF
END DO
i = minval_index
"""
parsed = processor.parse_fortran_statement(code)
processor.logger.info(parsed)

[INFO] Successfully parsed statement: minval_value = HUGE(0.0)
minval_index = 1
DO i = 1, N
  IF (A(i) .LT. minval_value) THEN
    minval_value = A(i)
    minval_index = i
  END IF
END DO
i = minval_index

[INFO] minval_value = HUGE(0.0)
minval_index = 1
DO i = 1, N
  IF (A(i) .LT. minval_value) THEN
    minval_value = A(i)
    minval_index = i
  END IF
END DO
i = minval_index

In [6]:
# A Fortran comment / compiler directives
comment = "!$ACC END PARALLEL"
comment_node = processor.parse_fortran_comment(comment)
processor.logger.info(comment_node)

[INFO] Successfully parsed comment: !$ACC END PARALLEL

[INFO] !$ACC END PARALLEL

In [7]:
# Benchmark directory is an attribute (set it accordingly)
processor.benchmark_dir = "/path/to/benchmark"
# Generate and parse a dummy subroutine named 'my_subroutine'
dummy_subroutine_node = processor.initiate_empty_routine("my_subroutine")
processor.logger.info(dummy_subroutine_node)

[INFO] SUBROUTINE read_dummy
  OPEN(UNIT = 1363, FILE = '/path/to/benchmark/my_subroutine/dummy.bin', FORM = 'unformatted', STATUS = 'old')
  WRITE(*, *) '--- inside the read_dummy routine for my_subroutine ---'
END SUBROUTINE read_dummy

In [8]:
# Sample combined declaration statement
stat = F23.Type_Declaration_Stmt("REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup, Fdn, Fab")
items = processor.separate_entity_declarations(stat)
for decl in items:
    processor.logger.info(decl)

[INFO] REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup

[INFO] REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fdn

[INFO] REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fab

In [9]:
# Define which variables are modified inside the proceduer
var_modif = ["Fup", "b", "c"]
# Modify the first declaration (Fup)
modified_entity_declaration = processor.add_entity_to_declaration(items[0], var_modif)
processor.logger.info(modified_entity_declaration)

[INFO] REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup, Fup_copy

In [10]:
# A compound allocate statement with multiple variables and a stat= option
allocate_stmts = F23.Allocate_Stmt("ALLOCATE(a(n), b(m), STAT = ier)")
# Separate the allocate statement into individual allocations
allocated_stmts = processor.separate_entity_allocation(allocate_stmts)
for item in allocated_stmts:
    processor.logger.info(item)

[INFO] Successfully generated allocation statements

[INFO] ALLOCATE(a(n), STAT = ier)

[INFO] ALLOCATE(b(m), STAT = ier)

In [11]:
# Define which variables are modified inside the proceduer
var_modif = ['a','b','c']
allocated_stmts = processor.add_entity_to_allocation(item,var_modif, openacc=True)
for alloc in allocated_stmts:
    processor.logger.info(alloc)

[INFO] Successfully parsed statement: IF (.NOT. ALLOCATED(b)) THEN
  ALLOCATE(b(m), STAT = ier)
END IF

[INFO] Successfully parsed statement: IF (.NOT. ALLOCATED(b_copy)) THEN
  ALLOCATE(b_copy(m), STAT = ier)
END IF

[INFO] Successfully generated allocation statements

[INFO] IF (.NOT. ALLOCATED(b)) THEN
  ALLOCATE(b(m), STAT = ier)
END IF

[INFO] IF (.NOT. ALLOCATED(b_copy)) THEN
  ALLOCATE(b_copy(m), STAT = ier)
END IF

In [12]:
# Explicit declaration with known shape
explicit_dec = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(kjpindex) :: evapot")
# Implicit declaration with assumed shape (:)
implicit_dec = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(:) :: ava")
# Map the shape from explicit to implicit
mapped = processor.map_declaration(implicit_dec, explicit_dec=explicit_dec)
processor.logger.info(mapped)

[INFO] Mapped declaration: REAL(KIND = r_std), DIMENSION(kjpindex) :: ava

[INFO] REAL(KIND = r_std), DIMENSION(kjpindex) :: ava

In [13]:
# Use the same implicit declaration
implicit_dec = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(:) :: ava")
# Directly specify dimensions as a string
dimensions = "kjpindex, nlev"
# Call map_declaration with explicit dimension override
mapped = processor.map_declaration(implicit_dec, dimensions=dimensions)
processor.logger.info(mapped)

[INFO] Mapped declaration: REAL(KIND = r_std), DIMENSION(kjpindex, nlev) :: ava

[INFO] REAL(KIND = r_std), DIMENSION(kjpindex, nlev) :: ava

In [14]:
# Original declaration and allocation as typically found in Fortran
declaration_stmt = F23.Type_Declaration_Stmt("REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:) :: a")
allocate_stmt = F23.Allocate_Stmt("ALLOCATE(a(n), STAT = ier)")
# Combine the two into a single fixed-size declaration
variable_declarations = [allocate_stmt, declaration_stmt]
combined = processor.combine_allocate_declaration(variable_declarations)
processor.logger.info(combined)

[INFO] Combined statement: REAL(KIND = r_std), DIMENSION(n) :: a

[INFO] REAL(KIND = r_std), DIMENSION(n) :: a

In [15]:
# Original statement with INTENT
declaration_stmt = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(n), INTENT(OUT) :: a")
# Remove INTENT attribute
cleaned_stmts = processor.remove_intent_and_save([declaration_stmt])
# processor.logger.info results
for stmt in cleaned_stmts:
    processor.logger.info(stmt)

[INFO] Successfully removed INTENT and SAVE attributes from statements

[INFO] REAL(KIND = r_std), DIMENSION(n) :: a

In [16]:
# Example 1: Compare two real-valued arrays named `a` and `a_cpu`
processor.check_point('a', 'a_copy', ['REAL', 'DIMENSION'])

# Example 2: Compare two logical scalar variables named `a` and `a_cpu`
processor.check_point('a', 'a_copy', ['LOGICAL'])

# Example 3 (optional extension): Compare two logical arrays
processor.check_point('a', 'a_copy', ['LOGICAL', 'DIMENSION'])

# Example 4 (default case): Compare two scalar real variables
processor.check_point('a', 'a_copy', ['REAL'])

[INFO] Successfully parsed statement: IF (ALL(a .EQ. a_copy)) THEN
  WRITE(*, *) 'Test passed: All elements in a_comp are equal to a_copy.'
ELSE
  WRITE(*, *) ''
  WRITE(*, *) 'Test failed: All elements in a_comp do not match a_copy.'
  WRITE(*, '(A, E25.16)') 'Maximum absolute error:', MAXVAL(ABS(a - a_copy))
  WRITE(*, '(A, 2E25.16)') 'Min and Max of a_comp:', MINVAL(a), MAXVAL(a)
  WRITE(*, '(A, 2E25.16)') 'Min and Max of a_copy:', MINVAL(a_copy), MAXVAL(a_copy)
  WRITE(*, *) ''
END IF

[INFO] Successfully parsed statement: IF (a .EQV. a_copy) THEN
  WRITE(*, *) 'LOGICAL EQV test passed: a_comp is equal to a_copy.'
ELSE
  WRITE(*, *) ''
  WRITE(*, *) 'LOGICAL EQV test failed: a_comp does not match a_copy.'
  WRITE(*, '(A, L1)') 'a_comp:', a
  WRITE(*, '(A, L1)') 'a_copy:', a_copy
  WRITE(*, *) ''
END IF

[INFO] Successfully parsed statement: IF (ALL(a .EQV. a_copy)) THEN
  WRITE(*, *) 'LOGICAL EQV test passed: All elements in a_comp are equal to a_copy.'
ELSE
  WRITE(*, *) ''
  WRITE(*, *) 'LOGICAL EQV test failed: Not all elements in a_comp match a_copy.'
  WRITE(*, *) ''
END IF

[INFO] Successfully parsed statement: IF (a .EQ. a_copy) THEN
  WRITE(*, *) 'Test passed: a_comp is equal to a_copy.'
ELSE
  WRITE(*, *) ''
  WRITE(*, *) 'Test failed: a_comp does not match a_copy.'
  WRITE(*, '(A, E25.16)') 'Absolute error:', ABS(a - a_copy)
  WRITE(*, '(A, 2E25.16)') 'a_comp:', a
  WRITE(*, '(A, 2E25.16)') 'a_copy:', a_copy
  WRITE(*, *) ''
END IF

Execution_Part(If_Construct(If_Then_Stmt(Level_4_Expr(Name('a'), '.EQ.', Name('a_copy'))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("'Test passed: a_comp is equal to a_copy.'", None),))), Else_Stmt(None), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("''", None),))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("'Test failed: a_comp does not match a_copy.'", None),))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Char_Literal_Constant("'(A, E25.16)'", None)))), Output_Item_List(',', (Char_Literal_Constant("'Absolute error:'", None), Intrinsic_Function_Reference(Intrinsic_Name('ABS'), Actual_Arg_Spec

In [17]:
# Create a CALL statement from a parsed subroutine tree
# This is useful when you want to generate a call to a subroutine whose AST you have already parsed.
call_stmt = processor.create_call_stmt(subroutine_tree)
processor.logger.info(call_stmt)

[INFO] CALL increment_counter(value)

## Extractor and Navigator classes testing
Some methods in the `Isolator` class depend on other classes that must be initialized before using.
Once set up, you can use and test individual methods of the class interactively in the notebook environment.

In [18]:
# Define the path to the Fortran module we want to analyze.
# This is the subdirectory (relative or absolute) where the source code resides.
rest_of_path = "tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"

# Define the name of the target Fortran module (without the .f90 extension).
target_module = "hydrol"

# Retrieve the base working directory from the environment variable `works`.
# This should be set in your shell or notebook environment beforehand.
work = os.getenv("works")

# Full path to the target module
full_module_path = os.path.join(work, rest_of_path, f"{target_module}.f90")

processor.logger.info(f"Full module path: {full_module_path}")

[INFO] Full module path: /scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90

In [19]:
# Create an instance of the Isolator and Extractor classes

from fgpt.isolator import Isolator
from fgpt.core.frontend.extractor import Extractor

# The Isolator class prepares the environment to isolate a Fortran procedure
# from a full codebase so that it can be run independently.
isolator = Isolator(
    rest_of_path=rest_of_path, 
    target_module=target_module, 
    work=work, 
    openacc=False,
    tapenade=False,
    f2py=False,
    py2jx=False
)

# The Extractor class takes the parsed Fortran module tree from the isolator
# and extracts all necessary metadata such as subroutine calls, variable declarations,
# dummy arguments, and other dependencies.
extractor = Extractor(
    isolator.module_dir_sp, 
    isolator.module_tree_cp,
    isolator.logger
)
extractor.find_subroutines()

╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: Isolator                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[INFO] Successfully parsed file: 
/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.fgpt

[INFO] Normalizing names in parsed file: 
/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.fgpt

[INFO] Successfully normalized all names in the module

[WARNING] ⚠ Subroutine 'hydrol_main' calls 'explicitsnow_main' which is not defined in current module

...

[INFO] Normalizing names in parsed file: 
/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow_org.fgpt

[INFO] Successfully normalized all names in the module

[INFO] ✅ Found module 'explicitsnow' in file: 
'/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow_org.fgpt'

[INFO] ✅ Found subroutine 'explicitsnow_main' in module 'explicitsnow', 
'/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow.f90'

[INFO] Found external subroutine 'explicitsnow_main' in file: explicitsnow_main, adding to processing queue

In [20]:
# Extract subroutines and gather their metadata
extractor.module_dir

'/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/'

In [21]:
extractor.actual_arg_spec_list['hydrol_soil_setup']

[['kjpindex', 'jst'], ['kjpindex', 'jst']]

In [22]:
extractor.dummy_arg_list['hydrol_soil_setup']

['kjpindex', 'ins']

In [23]:
extractor.call_subroutines['hydrol_soil_setup']

[Call_Stmt(Name('hydrol_soil_setup'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('jst')))),
 Call_Stmt(Name('hydrol_soil_setup'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('jst'))))]

In [24]:
# Extract subroutines and gather their metadata
extractor.find_subroutines()

# List all identified subroutine names in the module
extractor.subroutine_keys_all

[WARNING] ⚠ Subroutine 'hydrol_main' calls 'explicitsnow_main' which is not defined in current module

[INFO] 🔎 Searching for module name 'hydrol'

[INFO] ✅ Module 'hydrol' is already cached (path: 
'/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90')

[INFO] 🔎 Searching for module name 'explicitsnow'

[INFO] ✅ Module 'explicitsnow' is already cached (path: 
'/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow.f90')

[INFO] ✅ Found subroutine 'explicitsnow_main' in module 'explicitsnow', 
'/scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow.f90'

[INFO] Found external subroutine 'explicitsnow_main' in file: explicitsnow_main, adding to processing queue

{'explicitsnow_age',
 'explicitsnow_compactn',
 'explicitsnow_compactn_up',
 'explicitsnow_drift',
 'explicitsnow_fall',
 'explicitsnow_gone',
 'explicitsnow_grain',
 'explicitsnow_icelevels',
 'explicitsnow_icemelt',
 'explicitsnow_iceprofile',
 'explicitsnow_levels',
 'explicitsnow_main',
 'explicitsnow_maxmass',
 'explicitsnow_melt_refrz',
 'explicitsnow_profile',
 'explicitsnow_subli',
 'explicitsnow_transf',
 'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_muff',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_coef_setup',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr

In [25]:
level1_calls = """
call level1_sub(local_array)
"""
call_stmt = F23.Call_Stmt("call level1_sub(local_array)")
call_stmt

Call_Stmt(Name('level1_sub'), Actual_Arg_Spec_List(',', (Name('local_array'),)))

In [26]:
for key in extractor.call_within_sub['hydrol_main'].keys():
    processor.logger.info(f"Subroutine {key} is called:") 
    for call in extractor.call_within_sub['hydrol_main'][key]:
        processor.logger.info(f"     {call}")

[INFO] Subroutine hydrol_nudge_snow is called:

[INFO]      CALL hydrol_nudge_snow(kjit, kjpindex, snowdz, snowrho, snowtemp)

[INFO]      CALL hydrol_nudge_snow(kjit, kjpindex, snowdz, snowrho, snowtemp)

[INFO] Subroutine hydrol_hydraulic_arch_tuzet_calc is called:

[INFO]      CALL hydrol_hydraulic_arch_tuzet_calc(kjit, kjpindex, ks, nvan, avan, transpir, mc_out, veget, 
veget_max, njsc, soiltile, circ_class_n, circ_class_biomass, u, v, tq_cdrag, gsmean, pb, temp_air, lalo, psi_leaf, 
psi_leaf_next, psi_sto_leaf_save, psi_sto_wood_save, e_frac, psi_root_sup, psi_root_inf, psi_xylem_trunk, 
psi_xylem_leaf, psi_xylem_collar, psi_sto_wood, psi_sto_leaf, mc_i_sup, mc_i_inf, f_absorption)

...

[INFO]      CALL hydrol_alma(kjpindex, index, .FALSE., qsintveg, snow, snow_nobio, soilwet)

[INFO]      CALL hydrol_alma(kjpindex, index, .FALSE., qsintveg, snow, snow_nobio, soilwet)

[INFO] Subroutine hydrol_nudge_mc_diag is called:

[INFO]      CALL hydrol_nudge_mc_diag(kjpindex, soiltile)

[INFO]      CALL hydrol_nudge_mc_diag(kjpindex, soiltile)

In [27]:
# loops info
extractor.loop_dict

defaultdict(<function fgpt.core.frontend.extractor.Extractor.__init__.<locals>.<lambda>()>,
            {'hydrol_initialize': defaultdict(set, {'nslm': {'jsl'}}),
             'hydrol_main': defaultdict(set,
                         {'kjpindex': {'ji'},
                          'nvm': {'jv'},
                          'nstm': {'jst'},
                          'itopmax': {'jsl'},
                          'nslm': {'jsl'}}),
             'hydrol_init': defaultdict(set,
                         {'nslm': {'jsl'},
                          'nstm': {'jst'},
                          'kjpindex': {'ji'},
                          'nvm': {'jv'}}),
             'hydrol_tmc_update': defaultdict(set,
                         {'kjpindex': {'ji'},
                          'nvm': {'jv'},
                          'nstm': {'jst'},
                          'nslm - 1': {'jsl'},
                          'nslm': {'jsl'}}),
             'hydrol_var_init': defaultdict(set,
                         {'ns

In [28]:
# Call statement done within a subroutine
extractor.call_within_sub 
for parent in extractor.call_within_sub.keys():
    processor.logger.info('\n')
    processor.logger.info(f'Parent: {parent}, children: {extractor.call_within_sub[parent]}')

[INFO]

[INFO] Parent: hydrol_main, children: defaultdict(<class 'list'>, {'hydrol_nudge_snow': 
[Call_Stmt(Name('hydrol_nudge_snow'), Actual_Arg_Spec_List(',', (Name('kjit'), Name('kjpindex'), Name('snowdz'), 
Name('snowrho'), Name('snowtemp')))), Call_Stmt(Name('hydrol_nudge_snow'), Actual_Arg_Spec_List(',', (Name('kjit'),
Name('kjpindex'), Name('snowdz'), Name('snowrho'), Name('snowtemp'))))], 'hydrol_hydraulic_arch_tuzet_calc': 
[Call_Stmt(Name('hydrol_hydraulic_arch_tuzet_calc'), Actual_Arg_Spec_List(',', (Name('kjit'), Name('kjpindex'), 
Name('ks'), Name('nvan'), Name('avan'), Name('transpir'), Name('mc_out'), Name('veget'), Name('veget_max'), 
Name('njsc'), Name('soiltile'), Name('circ_class_n'), Name('circ_class_biomass'), Name('u'), Name('v'), 
Name('tq_cdrag'), Name('gsmean'), Name('pb'), Name('temp_air'), Name('lalo'), Name('psi_leaf'), 
Name('psi_leaf_next'), Name('psi_sto_leaf_save'), Name('psi_sto_wood_save'), Name('e_frac'), Name('psi_root_sup'), 
Name('psi_root_inf'), Name('psi_xylem_trunk'), Name('psi_xylem_leaf'), Name('psi_xylem_collar'), 
Name('psi_sto_wood'), Name('psi_sto_leaf'), Name('mc_i_sup'), Name('mc_i_inf'), Name('f_absorption')))), 
Call_Stmt(Name('hydrol_hydraulic_arch_tuzet_calc'), Actual_Arg_Spec_List(',', (Name('kjit'), Name('kjpindex'), 
Name('ks'), Name('nvan'), Name('avan'), Name('transpir'), Name('mc_out'), Name('veget'), Name('veget_max'), 
Name('njsc'), Name('soiltile'), Name('circ_class_n'), Name('circ_class_biomass'), Name('u'), Name('v'), 
Name('tq_cdrag'), Name('gsmean'), Name('pb'), Name('temp_air'), Name('lalo'), Name('psi_leaf'), 
Name('psi_leaf_next'), Name('psi_sto_leaf_save'), Name('psi_sto_wood_save'), Name('e_frac'), Name('psi_root_sup'), 
Name('psi_root_inf'), Name('psi_xylem_trunk'), Name('psi_xylem_leaf'), Name('psi_xylem_collar'), 
Name('psi_sto_wood'), Name('psi_sto_leaf'), Name('mc_i_sup'), Name('mc_i_inf'), Name('f_absorption'))))], 
'explicitsnow_main': [Call_Stmt(Name('explicitsnow_main'), Actual_Arg_Spec_List(',', (Name('kjpindex'), 
Name('precip_rain'), Name('precip_snow'), Name('temp_air'), Name('pb'), Name('u'), Name('v'), Name('temp_sol_new'),
Name('soilcap'), Name('pgflux'), Name('frac_nobio'), Name('totfrac_nobio'), Name('frac_snow_nobio'), Name('gtemp'),
Name('lambda_snow'), Name('cgrnd_snow'), Name('dgrnd_snow'), Name('contfrac'), Name('lambda_ice'), 
Name('cgrnd_ice'), Name('dgrnd_ice'), Name('ice_sheet_mask'), Name('vevapsno'), Name('snow_age'), 
Name('snow_nobio_age'), Name('snow_nobio'), Name('snowrho'), Name('snowgrain'), Name('snowdz'), Name('snowtemp'), 
Name('snowheat'), Name('snow'), Name('temp_sol_add'), Name('icetemp'), Name('icedz'), Name('snowliq'), 
Name('subsnownobio'), Name('grndflux'), Name('snowmelt'), Name('tot_melt'), Name('subsinksoil'), Name('zrainfall'),
Name('frac_snow_veg'), Name('veget'), Name('veget_max'), Name('run_off_lic'), Name('run_off_lic_frac')))), 
Call_Stmt(Name('explicitsnow_main'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('precip_rain'), 
Name('precip_snow'), Name('temp_air'), Name('pb'), Name('u'), Name('v'), Name('temp_sol_new'), Name('soilcap'), 
Name('pgflux'), Name('frac_nobio'), Name('totfrac_nobio'), Name('frac_snow_nobio'), Name('gtemp'), 
Name('lambda_snow'), Name('cgrnd_snow'), Name('dgrnd_snow'), Name('contfrac'), Name('lambda_ice'), 
Name('cgrnd_ice'), Name('dgrnd_ice'), Name('ice_sheet_mask'), Name('vevapsno'), Name('snow_age'), 
Name('snow_nobio_age'), Name('snow_nobio'), Name('snowrho'), Name('snowgrain'), Name('snowdz'), Name('snowtemp'), 
Name('snowheat'), Name('snow'), Name('temp_sol_add'), Name('icetemp'), Name('icedz'), Name('snowliq'), 
Name('subsnownobio'), Name('grndflux'), Name('snowmelt'), Name('tot_melt'), Name('subsinksoil'), Name('zrainfall'),
Name('frac_snow_veg'), Name('veget'), Name('veget_max'), Name('run_off_lic'), Name('run_off_lic_frac'))))], 
'hydrol_vegupd': [Call_Stmt(Name('hydrol_vegupd'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('veget'), 
Name('veget_max'), Name('soiltil

[INFO]

[INFO] Parent: hydrol_vegupd, children: defaultdict(<class 'list'>, {'hydrol_tmc_update': 
[Call_Stmt(Name('hydrol_tmc_update'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('veget_max'), 
Name('soiltile'), Name('qsintveg'), Name('drain_upd'), Name('runoff_upd')))), Call_Stmt(Name('hydrol_tmc_update'), 
Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('veget_max'), Name('soiltile'), Name('qsintveg'), 
Name('drain_upd'), Name('runoff_upd'))))]})

[INFO]

...

[INFO] Parent: hydrol_hydraulic_arch_tuzet_muff, children: defaultdict(<class 'list'>, 
{'hydrol_muff_radial_coef_setup': [Call_Stmt(Name('hydrol_muff_radial_coef_setup'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('igrid'), Name('ipft'), Name('njsc'), Name('ks'), Name('nvan'), Name('avan'), 
Name('lr_muff_sup'), Name('rad_sup'), Name('dri_sup'), Name('mc_i_sup_temp'), Name('f_sup_st'), Name('is_sup'), 
Name('tmat_rad'), Name('rhs_rad')))), Call_Stmt(Name('hydrol_muff_radial_coef_setup'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('igrid'), Name('ipft'), Name('njsc'), Name('ks'), Name('nvan'), Name('avan'), 
Name('lr_muff_inf'), Name('rad_inf'), Name('dri_inf'), Name('mc_i_inf_temp'), Name('f_inf_st'), Name('is_sup'), 
Name('tmat_rad'), Name('rhs_rad')))), Call_Stmt(Name('hydrol_muff_radial_coef_setup'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('igrid'), Name('ipft'), Name('njsc'), Name('ks'), Name('nvan'), Name('avan'), 
Name('lr_muff_sup'), Name('rad_sup'), Name('dri_sup'), Name('mc_i_sup_temp'), Name('f_sup_st'), Name('is_sup'), 
Name('tmat_rad'), Name('rhs_rad')))), Call_Stmt(Name('hydrol_muff_radial_coef_setup'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('igrid'), Name('ipft'), Name('njsc'), Name('ks'), Name('nvan'), Name('avan'), 
Name('lr_muff_inf'), Name('rad_inf'), Name('dri_inf'), Name('mc_i_inf_temp'), Name('f_inf_st'), Name('is_sup'), 
Name('tmat_rad'), Name('rhs_rad'))))]})

[INFO]

[INFO] Parent: hydrol_muff_radial_coef_setup, children: defaultdict(<class 'list'>, 
{'hydrol_muff_radial_resolution': [Call_Stmt(Name('hydrol_muff_radial_resolution'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('igrid'), Name('ipft'), Name('tmat_rad'), Name('rhs_rad'), Name('is_sup'), Name('mc_i')))),
Call_Stmt(Name('hydrol_muff_radial_resolution'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('igrid'), 
Name('ipft'), Name('tmat_rad'), Name('rhs_rad'), Name('is_sup'), Name('mc_i'))))]})

[INFO]

[INFO] Parent: explicitsnow_main, children: defaultdict(<class 'list'>, {'explicitsnow_fall': 
[Call_Stmt(Name('explicitsnow_fall'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('precip_snow'), 
Name('temp_air'), Name('u'), Name('v'), Name('snowrho'), Name('snowdz'), Name('snowheat'), Name('snowgrain'), 
Name('snowtemp'), Name('psnowhmass')))), Call_Stmt(Name('explicitsnow_fall'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('precip_snow'), Name('temp_air'), Name('u'), Name('v'), Name('snowrho'), Name('snowdz'), 
Name('snowheat'), Name('snowgrain'), Name('snowtemp'), Name('psnowhmass'))))], 'explicitsnow_levels': 
[Call_Stmt(Name('explicitsnow_levels'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snow_depth_tmp'), 
Name('snowdz')))), Call_Stmt(Name('explicitsnow_levels'), Actual_Arg_Spec_List(',', (Name('kjpindex'), 
Name('snow_depth_tmp'), Name('snowdz')))), Call_Stmt(Name('explicitsnow_levels'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('snow_depth_tmp'), Name('snowdz')))), Call_Stmt(Name('explicitsnow_levels'), 
Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snow_depth_tmp'), Name('snowdz'))))], 'explicitsnow_transf': 
[Call_Stmt(Name('explicitsnow_transf'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snowdz_old'), 
Name('snowdz'), Name('snowrho'), Name('snowheat'), Name('snowgrain')))), Call_Stmt(Name('explicitsnow_transf'), 
Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snowdz_old'), Name('snowdz'), Name('snowrho'), Name('snowheat'),
Name('snowgrain')))), Call_Stmt(Name('explicitsnow_transf'), Actual_Arg_Spec_List(',', (Name('kjpindex'), 
Name('snowdz_old'), Name('snowdz'), Name('snowrho'), Name('snowheat'), Name('snowgrain')))), 
Call_Stmt(Name('explicitsnow_transf'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snowdz_old'), 
Name('snowdz'), Name('snowrho'), Name('snowheat'), Name('snowgrain'))))], 'explicitsnow_compactn': 
[Call_Stmt(Name('explicitsnow_compactn'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snowtemp'), 
Name('snowrho'), Name('snowdz')))), Call_Stmt(Name('explicitsnow_compactn'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('snowtemp'), Name('snowrho'), Name('snowdz'))))], 'explicitsnow_compactn_up': 
[Call_Stmt(Name('explicitsnow_compactn_up'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snowtemp'), 
Name('snowrho'), Name('snowdz'), Name('snowliq')))), Call_Stmt(Name('explicitsnow_compactn_up'), 
Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('snowtemp'), Name('snowrho'), Name('snowdz'), 
Name('snowliq'))))], 'explicitsnow_drift': [Call_Stmt(Name('explicitsnow_drift'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('u'), Name('v'), Name('snowrho'), Name('snowdz')))), Call_Stmt(Name('explicitsnow_drift'), 
Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('u'), Name('v'), Name('snowrho'), Name('snowdz'))))], 
'explicitsnow_profile': [Call_Stmt(Name('explicitsnow_profile'), Actual_Arg_Spec_List(',', (Name('kjpindex'), 
Name('cgrnd_snow'), Name('dgrnd_snow'), Name('lambda_snow'), Name('temp_sol_new'), Name('snowtemp'), 
Name('snowdz'), Name('temp_sol_add')))), Call_Stmt(Name('explicitsnow_profile'), Actual_Arg_Spec_List(',', 
(Name('kjpindex'), Name('cgrnd_snow'), Name('dgrnd_snow'), Name('lambda_snow'), Name('temp_sol_new'), 
Name('snowtemp'), Name('snowdz'), Name('temp_sol_add'))))], 'explicitsnow_gone': 
[Call_Stmt(Name('explicitsnow_gone'), Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('pgflux'), 
Name('snowheat'), Name('snowtemp'), Name('snowdz'), Name('snowrho'), Name('snowliq'), Name('grndflux'), 
Name('snowmelt')))), Call_Stmt(Name('explicitsnow_gone'), Actual_Arg_Spec_List(',', (Name('kjpindex'), 
Name('pgflux'), Name('snowheat'), Name('snowtemp'), Name('snowdz'), Name('snowrho'), Name('snowliq'), 
Name('grndflux'), Name('snowmelt'))))], 'explicitsnow_melt_refrz': [Call_Stmt(Name('explicitsnow_melt_refrz'), 
Actual_Arg_Spec_List(',', (Name('kjpindex'), Name('precip_rain'), Name('pgflux'), Name('soilcap'), 
Name('totfr

Now, we examine the subroutines (children) that are called within other subroutines (parents). During isolation, the process begins with the most deeply nested child subroutines and proceeds outward, gradually isolating higher-level parent subroutines that depend on them.

In [29]:
# we will begin with seeing a subroutine of hydrol_soil
subroutine_key = 'hydrol_diag_soil'

In [30]:
# Retrieve the full AST node (subroutine as a Fparser Subroutine_Subprogram object)
subroutine_tree = extractor.subroutines[subroutine_key]

# Re-parse the subroutine into a new AST for further analysis or transformation
parsed_subroutine_tree = processor.parse_fortran_string(str(subroutine_tree))

# Analyze the usage of dummy arguments within a Fortran subroutine, None means that the variable is in the dummy, but not used. 
extractor.extract_intent(subroutine_key, subroutine_tree)
processor.logger.info(extractor.general_usage_dict[subroutine_key])
# Validates and corrects INTENT specifications
extractor.clean_subroutine(subroutine_key, subroutine_tree)

[INFO] Successfully parsed string!

[INFO] {'ks': 'IN', 'nvan': None, 'avan': None, 'mcr': None, 'mcs': 'IN', 'mcfc': 'IN', 'mcw': 'IN', 'kjpindex': 
'IN', 'veget_max': 'IN', 'soiltile': 'IN', 'njsc': None, 'runoff': 'INOUT', 'drainage': 'INOUT', 'evapot': None, 
'vevapnu': 'INOUT', 'returnflow': 'IN', 'reinfiltration': 'IN', 'irrigation': 'IN', 'shumdiag': 'INOUT', 
'shumdiag_perma': 'INOUT', 'k_litt': 'INOUT', 'litterhumdiag': 'INOUT', 'humrel': 'INOUT', 'vegstress': 'INOUT', 
'drysoil_frac': 'OUT', 'tot_melt': 'IN', 'us': 'INOUT', 'precip_rain': 'IN', 'totfrac_nobio': 'IN', 
'frac_snow_nobio': 'IN'}

[WARNING] ⚠ Name 'njsc' is not used. Declaration: INTEGER(KIND = i_std), DIMENSION(kjpindex), INTENT(IN) :: njsc

[WARNING] ⚠ Name 'evapot' is not used. Declaration: REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: evapot

[WARNING] ⚠ Name 'nvan' is not used. Declaration: REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: nvan

...

[WARNING] ⚠ Name 'vegstress', Expected: 'INOUT', Found: 'OUT'

[WARNING] ⚠ Original Declaration Statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(OUT) :: vegstress

[WARNING] ⚠ Modified Declaration Statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(INOUT) :: 
vegstress

[WARNING] ⚠ Expected exactly one Entity_Decl but found 5. Found: ['ji', 'jv', 'jsl', 'jst', 'i']. Breaking ...

[WARNING] ⚠ Expected exactly one Entity_Decl but found 2. Found: ['k_tmp', 'tmc_litter_ratio']. Breaking ...

In [31]:
# extract and categorize variables
extractor.find_variables(subroutine_tree,subroutine_key)

# processor.logger.info categorized variables for inspection
processor.logger.info("Declared Variables:")
processor.logger.info(extractor.var_declared[subroutine_key])

processor.logger.info("\n Dummy Arguments (with intent):")
for stmt in extractor.var_dummy[subroutine_key]:
    processor.logger.info(stmt.tostr())

processor.logger.info("\n Local Variables (non-dummy, declared):")
for stmt in extractor.var_local[subroutine_key]:
    processor.logger.info(stmt.tostr())

processor.logger.info("\n Global Variables (used but not declared):")
processor.logger.info(extractor.var_global[subroutine_key])

processor.logger.info("\n Modified Variables (on LHS of assignment):")
processor.logger.info(extractor.var_modif[subroutine_key])

processor.logger.info("\n Implicitly Shaped Variables (transformed to explicit):")
for name, decl in extractor.imp_shape[subroutine_key].items():
    processor.logger.info(f"{name} -> {decl.tostr()}")

[INFO] Declared Variables:

[INFO] {'us', 'precip_rain', 'frac_snow_nobio', 'soiltile', 'i', 'tmc_litter_ratio', 'kjpindex', 'litterhumdiag', 
'runoff', 'reinfiltration', 'shumdiag_perma', 'totfrac_nobio', 'veget_max', 'returnflow', 'jsl', 'vegstress', 
'mcr', 'humrel', 'drysoil_frac', 'nvan', 'k_tmp', 'evapot', 'shumdiag', 'ks', 'ji', 'k_litt', 'irrigation', 
'drainage', 'mask_vegtot', 'jv', 'vevapnu', 'tot_melt', 'njsc', 'mcfc', 'jst', 'avan', 'mcw', 'mcs'}

[INFO] 
 Dummy Arguments (with intent):

[INFO] REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: avan

[INFO] REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(INOUT) :: drainage

...

[INFO] 
 Global Variables (used but not declared):

[INFO] [Name('ae_ns'), Name('mask_soiltile'), Name('dr_ns'), Name('ru_ns'), Name('tmc'), Name('humrelv'), 
Name('mc'), Name('zero'), Name('humtot'), Name('tmc_litt_dry_mea'), Name('tmc_litt_wet_mea'), Name('tmc_litt_mea'),
Name('ok_freeze_cwrr'), Name('profil_froz_hydro'), Name('vegtot'), Name('min_sechiba'), Name('frac_bare_ns'), 
Name('profil_froz_hydro_ns'), Name('subsinksoil'), Name('iice'), Name('vegstressv'), Name('tmc_litter'), 
Name('tmc_litter_res'), Name('imin'), Name('tmc_litter_sat'), Name('imax'), Name('k_lin'), Name('soil_wet_litter'),
Name('tmc_litter_awet'), Name('tmc_litter_adry'), Name('un'), Name('soilmoist'), Name('dz'), Name('trois'), 
Name('huit'), Name('soilmoist_s'), Name('soilmoist_liquid'), Name('mcl'), Name('vegtot_old'), Name('soil_wet_ns'), 
Name('dh')]

[INFO] 
 Modified Variables (on LHS of assignment):

[INFO] set()

[INFO] 
 Implicitly Shaped Variables (transformed to explicit):

In [32]:
# Recursively searches for the declarations of global variables and external procedures 
extractor.find_global_variables(isolator.module_dir_sp, isolator.module_tree_cp, extractor.var_global[subroutine_key], subroutine_key)

[INFO] ⏳... Searching for variable 'ae_ns'

[INFO] 'ae_ns' is found in 'hydrol' of the module 'hydrol'

[INFO] REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:, :) :: ae_ns

[INFO] 'ae_ns' is found in 'hydrol_init' of the module 'hydrol'

[INFO] ALLOCATE(ae_ns(kjpindex, nstm), STAT = ier)

...

[INFO] REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:) :: dh

[INFO] 'dh' is found in 'hydrol_init' of the module 'hydrol'

[INFO] ALLOCATE(dh(nslm), STAT = ier)

[INFO] The containing directory is: /scratchu/kardaneh/tmp/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/

[INFO] ✅ Variable found!

In [33]:
# Process dummy argument declarations to extract shape and scalar information
extractor.process_declaration_variables(extractor.var_dummy[subroutine_key], subroutine_key)

# Process global declarations retrieved from other modules (already stored in dec_global)
for key in extractor.dec_global[subroutine_key].keys():
    extractor.process_declaration_variables(extractor.dec_global[subroutine_key][key], subroutine_key)

# Identify shape-related variables that still need to be searched in other modules
# We subtract scalar and already-known global variables from all shape variables
scalar_names = {var.tostr() for var in extractor.scalar_variables[subroutine_key]}
global_names = {var.tostr() for var in extractor.var_global[subroutine_key]}
shape_to_search = [
            var
            for var in extractor.shapes_variables[subroutine_key]
            if var.tostr() not in scalar_names and var.tostr() not in global_names
]

# If there are still unresolved shape variables, search them in other modules
if shape_to_search:
    extractor.find_global_variables(isolator.module_dir_sp, isolator.module_tree_cp, shape_to_search, subroutine_key)
    extractor.var_global[subroutine_key].extend(shape_to_search)


processor.logger.info(" Scalar Variables:")
processor.logger.info(extractor.scalar_variables[subroutine_key])

processor.logger.info("\n Shape Variables:")
processor.logger.info(extractor.shapes_variables[subroutine_key])

processor.logger.info("\n Variables to search in other modules (shape_to_search):")
processor.logger.info(shape_to_search)

processor.logger.info("\n Final Global Variables (after update):")
processor.logger.info(extractor.var_global[subroutine_key])

[INFO] ⏳... Searching for variable 'nnobio'

[INFO] 'nnobio' is not found in the current module. Searching in child modules...

[INFO] Module 'ioipsl' is added into the queue.

[INFO] Module 'xios_orchidee' is added into the queue.

[INFO] Module 'constantes' is added into the queue.

...

[INFO] [Name('nnobio'), Name('imin'), Name('imax')]

[INFO] 
 Variables to search in other modules (shape_to_search):

[INFO] [Name('nnobio')]

[INFO] 
 Final Global Variables (after update):

[INFO] [Name('ae_ns'), Name('mask_soiltile'), Name('dr_ns'), Name('ru_ns'), Name('tmc'), Name('humrelv'), 
Name('mc'), Name('zero'), Name('humtot'), Name('tmc_litt_dry_mea'), Name('tmc_litt_wet_mea'), Name('tmc_litt_mea'),
Name('ok_freeze_cwrr'), Name('profil_froz_hydro'), Name('vegtot'), Name('min_sechiba'), Name('frac_bare_ns'), 
Name('profil_froz_hydro_ns'), Name('subsinksoil'), Name('iice'), Name('vegstressv'), Name('tmc_litter'), 
Name('tmc_litter_res'), Name('imin'), Name('tmc_litter_sat'), Name('imax'), Name('k_lin'), Name('soil_wet_litter'),
Name('tmc_litter_awet'), Name('tmc_litter_adry'), Name('un'), Name('soilmoist'), Name('dz'), Name('trois'), 
Name('huit'), Name('soilmoist_s'), Name('soilmoist_liquid'), Name('mcl'), Name('vegtot_old'), Name('soil_wet_ns'), 
Name('dh'), Name('nnobio')]

In [34]:
# Extract array shape information for the current subroutine
# This includes arrays from:
#   - Global declarations (cls.dec_global)
#   - Dummy arguments (cls.var_dummy)
#   - Local variables (cls.var_local, handled internally)

extractor.extract_all_array_info(
    extractor.dec_global[subroutine_key],  # External/global declarations
    extractor.var_dummy[subroutine_key],   # Dummy argument declarations
    subroutine_key                   # Current subroutine identifier
)

# 
processor.logger.info(f"Array shape info for subroutine '{subroutine_key}':")
for var_name, dims in extractor.all_array_info[subroutine_key].items():
    processor.logger.info(f" - {var_name}:")
    for i, dim in enumerate(dims):
        processor.logger.info(f"    Dim {i+1}: Start = {dim['dim_str']}, End = {dim['dim_end']}")


processor.logger.info(f"Modified variable info for subroutine '{subroutine_key}':")
for var, info in extractor.var_modif_info[subroutine_key].items():
    processor.logger.info(f" - {var}: {info}")

[INFO] Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns

[INFO] Combined statement: INTEGER(KIND = i_std), DIMENSION(kjpindex, nstm) :: mask_soiltile

[INFO] Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: dr_ns

[INFO] Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ru_ns

[INFO] Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tmc

...

[INFO]  - vevapnu:

[INFO]     Dim 1: Start = 1, End = kjpindex

[INFO]  - mask_vegtot:

[INFO]     Dim 1: Start = 1, End = kjpindex

[INFO] Modified variable info for subroutine 'hydrol_diag_soil':

In [35]:
# Extract the vectorized loop from the subroutine (if it uses 'kjpindex' as loop upper bound)
extractor.extract_loop_vect(subroutine_key, subroutine_tree)

# processor.logger.info the extracted vector loop structure for inspection
if extractor.loop_vect[subroutine_key]:
    processor.logger.info(f"Extracted vector loop for '{subroutine_key}':\n{extractor.loop_vect[subroutine_key]}")
else:
    processor.logger.info(f"No vector loop (ending with 'kjpindex') found in subroutine '{subroutine_key}'.")

[INFO] Extracted vector loop for 'hydrol_diag_soil':
DO ji = 1, kjpindex